In [ ]:
binary = True

In [ ]:

n_iter_DecisionTreeClassifier=50
n_iter_RandomForestClassifier=50
n_iter_GradientBoostingClassifier=50
n_iter_XGBClassifier=50

# Modelos de Classificação

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools
import seaborn as sns

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, cross_validate, train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
# from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score, confusion_matrix
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

In [ ]:
ind_vars = pd.read_csv(r'..\code_metrics_professor_util.csv', index_col='question')
ind_vars = ind_vars.fillna(0)

ind_vars

In [ ]:

dep_vars = pd.read_csv(r'Correlations_CSV\question_info.csv', index_col='question')

dep_vars = dep_vars.fillna(0)
dep_vars

In [ ]:
ind_vars = ind_vars.astype(np.float64)
dep_vars = dep_vars.astype(np.float64)

In [ ]:
dep_vars['taxa de erro'].describe()

In [ ]:
# Função para classificação ternária
def ternary_classify(rows, bounds, reverse=False):
    if len(bounds) != 2:
        raise Exception('quartiles must have 2 values, {} were given'.format(len(bounds)))
        
    values = ('facil', 'medio', 'dificil')
    if reverse:
        values = values[::-1]
    return rows.apply(lambda row: 
                      values[0] if row <= bounds[0] else
                      values[1] if row <= bounds[1] else
                      values[2]
                     )
    
def get_bounds_ternary(rows, column_name):
    if column_name == 'taxa de erro':
        # Classificação do INEP
        q1 = 20.0  # Primeiro quartil
        q3 = 40.0  # Terceiro quartil
        return (q1, q3)
    else:
        return np.quantile(rows, q=[1/3, 2/3], method='midpoint')

In [ ]:
# Classificação binária
def binary_classify(rows, bounds, reverse=False, custom_labels=None):
    """
    Função de classificação binária com rótulos padrão opcionais.
    
    :param rows: Dados a serem classificados
    :param bounds: Limites de classificação
    :param reverse: Se deve inverter as classificações
    :param custom_labels: Lista com dois rótulos alternativos, caso necessário
    :return: Série com classificação
    """
    if len(bounds) != 1:
        raise Exception('bounds deve conter exatamente 1 valor, {} foram fornecidos'.format(len(bounds)))
        
    # Rótulos padrão ou personalizados
    values = custom_labels if custom_labels else ('facil', 'dificil')
    
    if reverse:
        values = values[::-1]
    
    return rows.apply(lambda row: 
                      values[0] if row <= bounds[0] else
                      values[1]
                     )
    
def get_bounds_binary(rows, column_name):
    if column_name == 'taxa de erro':
        # Classificação do INEP
        return (40.0,)
    elif column_name == 'discriminacao':
        # Classificação do INEP
        return (0.09,)
    else:
        return np.quantile(rows, q=[0.5], method='midpoint')

In [ ]:
classified = pd.DataFrame(index=dep_vars.index, columns=dep_vars.columns)
bounds = {}

for col in classified.columns:
    if col == 'discriminacao':
        # Lógica binária para 'discriminacao'
        bounds[col] = get_bounds_binary(dep_vars[col], col)
        classified[col] = binary_classify(dep_vars[col], bounds[col], reverse=False, custom_labels=('facil', 'dificil'))
 
    else:
        if binary:  # Usa lógica binária para outras colunas se binary = True
            bounds[col] = get_bounds_binary(dep_vars[col], col)
            classified[col] = binary_classify(dep_vars[col], bounds[col], reverse=False)
        else:  # Usa lógica ternária para outras colunas se binary = False
            bounds[col] = get_bounds_ternary(dep_vars[col], col)
            classified[col] = ternary_classify(dep_vars[col], bounds[col])

# Resultado final
classified


In [ ]:
classified['discriminacao'].value_counts()

In [ ]:
classified['taxa de erro'].value_counts()

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Exemplo de carregamento dos dados (use seu DataFrame real)
# classified = pd.read_csv('seu_arquivo.csv')

# Contando a frequência de todas as categorias na coluna 'taxa de erro'
unique_dificuldade = classified['taxa de erro'].value_counts()

# Criar diretório para salvar o gráfico
caminho = r'Figures\Classification'
os.makedirs(caminho, exist_ok=True)

# Criar o gráfico
plt.figure(figsize=(6, 6))  # Ajuste o tamanho do gráfico
sns.barplot(
    x=unique_dificuldade.index,
    y=unique_dificuldade.values,
    edgecolor=".3",  # Bordas nas barras
    color="skyblue"  # Cor única
)
plt.xlabel('Classificação')
plt.ylabel('Frequência')
plt.title('Distribuição de Classificação')

# Salvar o gráfico
fig_name = "Classificacao_TaxaDeErro.png"
plt.tight_layout()
plt.savefig(os.path.join(caminho, fig_name))

# Mostrar o gráfico
plt.show()


In [ ]:
classified

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Exemplo de carregamento dos dados (use seu DataFrame real)
# classified = pd.read_csv('seu_arquivo.csv')

# Configuração binária ou ternária

# Reordenando e filtrando as categorias
ordem_desejada = ["facil", "dificil"]
grafico = classified[classified['discriminacao'].isin(ordem_desejada)]  # Filtrar apenas categorias desejadas
grafico['discriminacao'] = pd.Categorical(grafico['discriminacao'], categories=ordem_desejada, ordered=True)

# Renomeando as categorias conforme desejado
rename_map = {"facil": "muito fraco", "dificil": "fraco"}
grafico['discriminacao'] = grafico['discriminacao'].map(rename_map)

# Contando as frequências corretamente
unique_dificuldade = grafico['discriminacao'].value_counts().sort_index()

# Criar diretório para salvar o gráfico
caminho = r'Figures\Classification\2' if binary else r'Figures\Classification\3'
os.makedirs(caminho, exist_ok=True)

# Configurar os rótulos para binário ou ternário
xtick_labels = unique_dificuldade.index  # Agora com os nomes renomeados

# Criar o gráfico
plt.figure(figsize=(6, 6))  # Ajuste o tamanho do gráfico
sns.barplot(
    x=unique_dificuldade.index,
    y=unique_dificuldade.values,
    edgecolor=".3",  # Bordas nas barras
    color="skyblue"  # Cor única
)
plt.xlabel('Classificação')
plt.ylabel('Frequência')
plt.xticks(ticks=range(len(xtick_labels)), labels=xtick_labels)
plt.title('Distribuição de Classificação - Discriminacao')

# Salvar o gráfico
fig_name = "Classificacao_Discriminacao.png"
plt.tight_layout()
plt.savefig(os.path.join(caminho, fig_name))

# Mostrar o gráfico
plt.show()



rename_map = { "muito fraco":"facil",  "fraco":"dificil"}
grafico['discriminacao'] = grafico['discriminacao'].map(rename_map)


In [ ]:
classified

In [ ]:
scoring = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro','f1_micro']

In [ ]:
encoder = LabelEncoder()
#classes = ['facil', 'dificil'] if binary else ['facil', 'medio', 'dificil']
classes =  ['facil', 'medio', 'dificil']
encoder.fit(classes)

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, accuracy_score
from imblearn.over_sampling import SMOTE
import numpy as np


def cross_val_train(ind_vars, y, model, binary_class, is_encoded):
    """
    Função de validação cruzada ajustada com lógica específica para 'discriminacao' e a escolha de classes
    em função do parâmetro `binary_class`.
    """
    predicted_list = np.array([], dtype=int)
    tested_list = np.array([], dtype=int)
    kf = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

    for train_index, test_index in kf.split(ind_vars, y):
        X_train, X_test = ind_vars.iloc[train_index], ind_vars.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        # Aplicar oversampling apenas para a coluna 'taxa de erro'
        if y.name == 'taxa de erro':
            oversample = SMOTE(random_state=42)
            X_train, y_train = oversample.fit_resample(X_train, y_train)

        # Ajustar o modelo com os dados de treino
        model.fit(X_train, y_train)

        # Fazer previsões
        y_pred = model.predict(X_test)

        # Armazenar previsões e dados reais
        tested_list = np.append(tested_list, y_test)
        predicted_list = np.append(predicted_list, y_pred)

    # Descodificar previsões caso seja necessário
    if is_encoded:
        tested_list = encoder.inverse_transform(tested_list)
        predicted_list = encoder.inverse_transform(predicted_list)

    # Lógica para definir as classes usadas na métrica
    if y.name == 'discriminacao':
        # Sempre usar apenas as classes 'facil' e 'dificil' para discriminacao
        classes = ('facil', 'dificil')
        
    else:
        # Caso contrário, determinar as classes dependendo de binary_class
        classes = ('facil', 'dificil') if binary_class else ('facil', 'medio', 'dificil')

    try:
        # Calcular a matriz de confusão com as classes apropriadas
        cnf_matrix = confusion_matrix(tested_list, predicted_list, labels=classes)
    except ValueError as e:
        raise ValueError(
            f"Erro ao calcular a matriz de confusão.\n"
            f"Variável 'y_true' (tested_list): {np.unique(tested_list)}\n"
            f"Variável 'y_pred' (predicted_list): {np.unique(predicted_list)}\n"
            f"Classes esperadas: {classes}\n"
            f"Mensagem original: {str(e)}"
        )

    # Calcular as métricas de desempenho
    precision, recall, f1_macro, _ = precision_recall_fscore_support(
        tested_list, predicted_list, average='macro'
    )
    
    # Cálculo adicional da métrica f1_micro
    precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(
        tested_list, predicted_list, average='micro'
    )
    
    acc = accuracy_score(tested_list, predicted_list)

    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1_macro': f1_macro,
        'f1_micro': f1_micro,  # Incluindo f1_micro nos resultados
    }, cnf_matrix




In [ ]:
def model_train_search_cv(ind_vars, dep_vars, model, model_class, distributions, n_iter, train_type=cross_val_train,
                           binary_class=False, encode=False):
    scoring = ['accuracy', 'precision', 'recall', 'f1_macro', 'f1_micro']  # Adicionei 'f1_micro' aqui
    results = pd.DataFrame(index=dep_vars.columns, columns=scoring)
    cnf_matrixes = dict()
    
    for metric in results.index:
        pipeline_ls = []
        if metric == 'taxa de erro':
            pipeline_ls.append(("smote", SMOTE()))
        pipeline_ls.append(("model", model))

        pipe_distributions = {f'model__{key}': value for key, value in distributions.items()}
        pipeline = Pipeline(pipeline_ls)
        
        ndf = pd.Series(encoder.transform(dep_vars[metric]), name=metric, dtype=int) if encode else dep_vars[metric]
        cross_val = StratifiedKFold(n_splits=4, shuffle=True)

        # Mantendo a lógica do search com a métrica f1_macro
        clf = RandomizedSearchCV(
            estimator=pipeline,
            param_distributions=pipe_distributions,
            random_state=42,
            cv=cross_val,
            n_iter=n_iter,
            n_jobs=-1,
            scoring='f1_macro',
            return_train_score=True,
        )
        
        # Realiza a busca no modelo
        search = clf.fit(ind_vars, ndf)
        best_params = {key.split('__')[-1]: value for key, value in search.best_params_.items()}
        print('best params for {}: {}'.format(metric, best_params))
        best_model = model_class(**best_params)

        # Rodando a função de validação cruzada ajustada
        metric_train_result, cnf_matrix = train_type(ind_vars, ndf, best_model, binary_class, is_encoded=encode)
        
        # Salvando resultados na tabela
        results.loc[metric] = metric_train_result
        cnf_matrixes[metric] = cnf_matrix

    return results.sort_values(by=['f1_micro'], ascending=False), cnf_matrixes

## Matriz de Confusão

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import itertools

def plot_confusion_matrix(cnf_matrix, title,  base_dir="Figures", regression_dir="Classification", discriminacao=False):
    """
    Plota a matriz de confusão, exibe o gráfico e o salva em uma pasta específica baseada no valor de `binary` e `discriminacao`.

    Parameters:
        cnf_matrix (array-like): A matriz de confusão.
        title (str): O título do gráfico.
        binary (bool): Define a subpasta e o nome do arquivo. True -> Regression/2, False -> Regression/3.
        base_dir (str): O diretório base para salvar os gráficos. Padrão é "Figures".
        regression_dir (str): O subdiretório base "Regression".
        discriminacao (bool): Define se as classes são 'dificil' e 'facil'. Substitui o valor de `binary` se True.
    """
    # Configurar as classes com base nos parâmetros
    if discriminacao:
        classes = ['muito fraco', 'fraco']
        sub_dir = os.path.join(regression_dir, "discriminacao")
        file_name = "matriz_confusao_discriminacao.png"
    else:
        classes = ['fácil', 'difícil'] if binary else ['fácil', 'médio', 'difícil']
        sub_dir = os.path.join(regression_dir, "2" if binary else "3")
        file_name = f"matriz_confusao_{'2' if binary else '3'}.png"

    # Criar o caminho completo
    full_sub_dir = os.path.join(base_dir, sub_dir)
    os.makedirs(full_sub_dir, exist_ok=True)
    
    file_path = os.path.join(full_sub_dir, file_name)

    # Configurar o estilo e criar o gráfico
    plt.figure()
    plt.style.use('default')
    plt.imshow(cnf_matrix, interpolation='nearest', cmap=plt.get_cmap('Blues'))
    plt.title(title)
    plt.colorbar()

    # Configurar os ticks e rótulos
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    # Adicionar valores na matriz
    fmt = 'd'
    thresh = cnf_matrix.max() / 2.
    for i, j in itertools.product(range(cnf_matrix.shape[0]), range(cnf_matrix.shape[1])):
        plt.text(j, i, format(cnf_matrix[i, j], fmt), horizontalalignment="center",
                 color="white" if cnf_matrix[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('Classe predita')
    plt.xlabel('Classe verdadeira')

    # Salvar o gráfico no arquivo especificado
    plt.savefig(file_path, bbox_inches='tight')

    # Informar onde o arquivo foi salvo
    print(f"Gráfico salvo em: {file_path}")

    plt.show()


## Experimentos

In [ ]:
def push_best_results(best_results, df_results: pd.DataFrame, model):
    for dep_var, metrics in df_results.iterrows():
        acc_cur, f1_cur, _ = best_results[dep_var]
        acc_nxt, f1_nxt = df_results.loc[dep_var, ['accuracy', 'f1_macro']]  # Removendo f1_micro
        if f1_nxt > f1_cur or (np.abs(f1_nxt - f1_cur) < 1e-6 and acc_nxt > acc_cur):
            best_results[dep_var] = acc_nxt, f1_nxt, model


In [ ]:
columns = dep_vars.columns
best_results = dict()
for col in columns:
    best_results[col] = (0.0, 0.0, '')

### Árvore de Decisão

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
criterion = ['gini', 'entropy', 'log_loss']
splitter = ['best', 'random']
random_state = [42, 0, 7, 21, 15,19,12,4,17] 
distributions = dict(criterion=criterion, splitter=splitter, random_state=random_state)

In [ ]:
ind_vars

In [ ]:
classified

In [ ]:
tree = DecisionTreeClassifier()
scores, cnf_matrixes_dt = model_train_search_cv(ind_vars, classified, tree, DecisionTreeClassifier, distributions, n_iter_DecisionTreeClassifier, binary_class=binary)
push_best_results(best_results, scores, 'Árvore de Decisão')
scores

### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
n_estimators = np.linspace(start=100, stop=1000, num=10, dtype=int)
criterion = ['gini', 'entropy', 'log_loss']
class_weight = ['balanced', 'balanced_subsample', None]
random_state = [42]
distributions = dict(
    n_estimators=n_estimators,
    criterion=criterion,
    class_weight=class_weight,
    random_state=random_state
)

In [ ]:
rd_forest = RandomForestClassifier()


scores, cnf_matrixes_rf = model_train_search_cv(ind_vars, classified, rd_forest, RandomForestClassifier, distributions, n_iter_RandomForestClassifier, binary_class=binary)
push_best_results(best_results, scores, 'Random Forest')
scores

In [ ]:
plot_confusion_matrix(cnf_matrixes_rf['taxa de erro'], '')

In [ ]:
plot_confusion_matrix(cnf_matrixes_rf['discriminacao'], '',discriminacao=True)

### GradientBoosting

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

In [ ]:
gboost = GradientBoostingClassifier()

In [ ]:
loss = ['log_loss']
if binary:
    loss = ['log_loss', 'exponential']

learning_rate = np.linspace(start=0.01, stop=0.5, num=10)
n_estimators = np.linspace(start=1, stop=500, num=10, dtype=int)
criterion = ['friedman_mse', 'squared_error']
random_state = [42]

distributions = dict(
    loss = loss,
    learning_rate = learning_rate,
    n_estimators = n_estimators,
    criterion = criterion,
    random_state=random_state,
)

In [ ]:
scores, cnf_matrixes_gb = model_train_search_cv(ind_vars, classified, model=gboost, model_class=GradientBoostingClassifier, distributions=distributions, n_iter=n_iter_GradientBoostingClassifier, binary_class=binary)
push_best_results(best_results, scores, 'Gradient Boosting')
scores

### Extreme Gradient Boosting

In [ ]:
xgb = XGBClassifier()

In [ ]:
n_estimators = np.linspace(start=100, stop=500, num=10, dtype=int)
max_depth = np.linspace(start=3, stop=15, num=15, dtype=int)
eta = [0.0001, 0.001, 0.01, 0.1, 1.0]
subsample = np.linspace(start=0.1, stop=1.0, num=10, dtype=np.float64)
random_state = [42]

distributions = dict(n_estimators=n_estimators, max_depth=max_depth, eta=eta, subsample=subsample, random_state=random_state)
distributions

In [ ]:
scores, cnf_matrixes_xgb = model_train_search_cv(ind_vars, classified, model=xgb, model_class=XGBClassifier, distributions=distributions, n_iter=n_iter_XGBClassifier, binary_class=binary, encode=True)
push_best_results(best_results, scores, 'Extreme Gradient Boosting')
scores

In [ ]:
best_results

In [ ]:
best_results_df = pd.DataFrame(best_results, index=['f1_micro', 'accuracy', 'classificador']) \
    .T.sort_values(by=['f1_micro', 'accuracy'], ascending=False)
best_results_df

In [ ]:
pasta = "Classification_CSV/3" if not binary else "Classification_CSV/2"
os.makedirs(pasta, exist_ok=True)

caminho = os.path.join(pasta, "results_classification.csv")

best_results_df.to_csv(caminho, index=False)

In [ ]:
plot_confusion_matrix(cnf_matrixes_rf['taxa de erro'], '')

In [ ]:
plot_confusion_matrix(cnf_matrixes_rf['discriminacao'], 'Matriz de Confusão - Discriminação', discriminacao=True)

## Reduzindo número de variáveis independentes

In [ ]:

caminho=r'Correlations_CSV\best_code_metrics.csv'






best_code_metrics = pd.read_csv(caminho, index_col='metric')
code_attrs = pd.DataFrame(index=best_code_metrics.index, columns=best_code_metrics.columns)
for row in best_code_metrics.iterrows():
    cols = row[1].apply(lambda item: eval(item)[1])
    code_attrs.loc[row[0],:] = cols
code_attrs

## Selecionando métricas mais comuns

In [ ]:
attrs = {}
for row in code_attrs.iterrows():
    for key in row[1]:
        if not key in attrs:
            attrs[key] = 0
        attrs[key] += 1
best_attrs = list(attrs.items())
best_attrs.sort(key=lambda key: key[1], reverse=True)
best_attrs = list(map(lambda x: x[0], best_attrs[:10]))
best_attrs

In [ ]:
filtered_classified = ind_vars[best_attrs] 
filtered_classified